# LLM4Series — Evaluating Forecast Accuracy

This notebook covers the evaluation API of `llm4series`.
It shows how to compute SMAPE, MAE, and RMSE for both univariate and
multivariate series, how to call metrics individually or all at once,
and how to use the standalone `ls.metrics` function with plain DataFrames and lists.

## Step 1 — Import the library

In [1]:
import llm4series as ls

## Step 2 — Load a univariate series and create a dummy prediction

A simple 10 % over-prediction is used as a stand-in for a real model forecast.
This lets us verify that metrics are computed correctly with a known offset.

In [2]:
ts = ls.read_file("data/OT.csv")
prediction = ts * 1.1  # 10 % over-prediction as a stand-in for a real forecast

## Step 3 — Compute all metrics at once with `.metrics()`

`ts.metrics(prediction)` returns a DataFrame with SMAPE, MAE, and RMSE in a single call.
For a univariate series the result has one column named after the variable.

In [3]:
ts.metrics(prediction)

,OT
smape,9.52
mae,2.66
rmse,2.91


## Step 4 — Compute individual metrics

Each metric is also available as a standalone method: `.smape()`, `.mae()`, `.rmse()`.
This is useful when you need only one metric or want to build a custom summary.

In [4]:
print(ts.smape(prediction), ts.mae(prediction), ts.rmse(prediction))

9.52 2.66 2.91


## Step 5 — Control decimal precision

The `decimals` parameter rounds the result to a given number of decimal places.
This is useful when logging results or building leaderboard tables.

In [5]:
ts.smape(prediction, decimals=1)  # Round to one decimal place

np.float64(9.5)

## Step 6 — Load a multivariate series and create a dummy prediction

The same pattern applies to `MultiTimeSeries`. Each column is evaluated independently.

In [6]:
ts = ls.read_file("data/ETTh2.csv")
prediction = ts * 1.1

## Step 7 — All metrics for a multivariate series

`.metrics()` now returns a DataFrame with one column per variable
and one row per metric, making it easy to compare accuracy across variables.

In [7]:
ts.metrics(prediction)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
smape,9.49,7.43,9.52,8.94,8.87,6.36,9.52
mae,3.72,0.87,4.38,0.84,0.60,0.26,2.66
rmse,3.86,1.04,4.57,0.94,0.70,0.64,2.91


## Step 8 — Individual metrics for a multivariate series

Individual metric methods return a pandas Series indexed by variable name,
so you can rank or filter variables by their error.

In [8]:
print(ts.smape(prediction), ts.mae(prediction), ts.rmse(prediction))

HUFL    9.49
HULL    7.43
MUFL    9.52
MULL    8.94
LUFL    8.87
LULL    6.36
OT      9.52
dtype: float64 HUFL    3.72
HULL    0.87
MUFL    4.38
MULL    0.84
LUFL    0.60
LULL    0.26
OT      2.66
dtype: float64 HUFL    3.86
HULL    1.04
MUFL    4.57
MULL    0.94
LUFL    0.70
LULL    0.64
OT      2.91
dtype: float64


## Step 9 — Rounding also works for multivariate series

`decimals=1` works the same way: each per-variable value is rounded.

In [9]:
ts.smape(prediction, decimals=1)

HUFL    9.5
HULL    7.4
MUFL    9.5
MULL    8.9
LUFL    8.9
LULL    6.4
OT      9.5
dtype: float64

## Step 10 — Standalone `ls.metrics` with plain DataFrames and lists

`ls.metrics` is a module-level function that accepts plain pandas DataFrames
or Python lists — no `UniTimeSeries` or `MultiTimeSeries` required.
This makes it easy to evaluate any external model output without wrapping it first.

- When passed DataFrames, it returns a DataFrame with variables as rows.
- When passed lists (a single variable), it returns a single-column DataFrame.

In [10]:
import pandas as pd

# Multivariate: compare two DataFrames
df1 = pd.DataFrame(ts * 1.1)
df2 = pd.DataFrame(ts * 1.2)
display(ls.metrics(df1, df2))

# Univariate: compare two plain Python lists
list1 = list(ts["OT"] * 1.1)
list2 = list(ts["OT"] * 1.2)
display(ls.metrics(list1, list2))

,smape,mae,rmse
HUFL,8.67,3.72,3.86
HULL,6.78,0.87,1.04
MUFL,8.70,4.38,4.57
MULL,8.16,0.84,0.94
LUFL,8.10,0.60,0.70
LULL,5.80,0.26,0.64
OT,8.70,2.66,2.91


,value
smape,8.70
mae,2.66
rmse,2.91
